# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete guide for loading and exploring a Croissant-compliant dataset with the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

We will:
- Load the dataset metadata and records.
- Review record sets and their field `@id`s.
- Load data for specific record sets.
- Process, analyze, and visualize data using pandas and matplotlib/seaborn.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and explore high-level dataset attributes using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display name and description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review all available record sets. Each record set, field, and column is referenced by their `@id` as per the Croissant specification.

In [ ]:
# List all record sets and their available fields, referencing by @id.
print("Record sets and their fields (by @id):\n")
all_record_sets = list(dataset.record_sets())

if not all_record_sets:
    print("No record sets found in dataset metadata.\n")
else:
    for rec in all_record_sets:
        print(f"Record set: {rec['@id']}")
        fields = rec.get('fields', [])
        if fields:
            for field in fields:
                print(f"  Field: {field['@id']}")
        else:
            print("  No fields found.")
        print("")

# If the dataset does not expose record_sets() (older mlcroissant), fallback:
if not all_record_sets:
    # mlcroissant >=1.0: Use .record_set_ids
    try:
        print("Fallback: enumerate available record set IDs using dataset.record_set_ids()")
        print("Available record set @id's:")
        for rsid in dataset.record_set_ids():
            print(f"  {rsid}")
    except AttributeError:
        print("No record sets found, unable to enumerate.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. **All references must be by canonical `@id`**. List available record sets and pick one for analysis.

In [ ]:
# Get all the available record set @id's
try:
    record_sets_ids = list(dataset.record_set_ids())
except Exception as e:
    # record_set_ids() is >=mlcroissant-1.0, fallback by inspecting metadata
    meta = dataset.metadata._data
    if "recordSet" in meta:
        record_sets_ids = []
        rs_list = meta["recordSet"]
        if isinstance(rs_list, dict):
            record_sets_ids = [rs_list.get("@id")]
        elif isinstance(rs_list, list):
            record_sets_ids = [rset.get("@id") for rset in rs_list if "@id" in rset]
        else:
            record_sets_ids = []
    else:
        # Sometimes the Dataset object auto-discovers single record set from distribution
        record_sets_ids = []

print(f"Available record set IDs: {record_sets_ids}")

# If no explicit record sets, try main dataset @id as a record set
if not record_sets_ids:
    main_dataset_id = dataset.metadata['@id'] if hasattr(dataset.metadata, '__getitem__') else getattr(dataset.metadata, '@id', None)
    if main_dataset_id:
        record_sets_ids = [main_dataset_id]
        print(f"Only one record set found (main dataset @id): {main_dataset_id}")

# For this dataset, as per the Croissant payload, the main dataset @id is used as a record set.
if not record_sets_ids:
    record_sets_ids = ['https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0']
    print(f"Imported fallback record set @id: {record_sets_ids}")

# Extract records for each record set
dataframes = {}
for recset_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded DataFrame for record set @id: {recset_id}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {recset_id}: {e}")

# Preview data for the first record set
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nPreview of DataFrame for: {first_rs}\n")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
We apply common data processing steps. We use **field and record set `@id` references** as required. Let's select a numeric field and group/categorize by another field if possible.

In [ ]:
import numpy as np

# Assign the record set @id you want to analyze
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]
print(f"Analyzing record set: {record_set_id}")
print(f"Fields (columns): {df.columns.tolist()}")

# Try to pick a numeric column (by inspection or known schema). Here we try several common names:
possible_numeric_fields = [
    '@id:coverage', '@id:peptide_count', '@id:MW', '@id:pI',
    'Coverage', 'Normalized Abundance', 'MW', 'peptide_count', 'pI', 'Abundance'
]
numeric_field = None
for f in possible_numeric_fields:
    if f in df.columns and np.issubdtype(df[f].dtype, np.number):
        numeric_field = f
        break
# If scan did not succeed, find any numeric column
if not numeric_field:
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break

if numeric_field:
    print(f"Selected numeric field: {numeric_field}")
else:
    print("No numeric field found. Please inspect DataFrame and adjust field selection.")
    numeric_field = df.columns[0]

# Filter records with numeric_field > threshold
threshold = df[numeric_field].mean() if numeric_field else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Records with {numeric_field} > {threshold}")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field for filtered records
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a categorization field (try 'Description', 'accession', 'Protein', etc.)
group_field = None
for col in ['@id:description', 'Description', 'Protein', 'Gene', 'accession', 'Category']:
    if col in df.columns:
        group_field = col
        break

if group_field:
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No suitable group_field found. EDA by group skipped.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, optionally, its values grouped by another field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping is possible, show top 10 groups by mean numeric_field
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10,5))
    s = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    sns.barplot(data=s, x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded a FAIR^2 Croissant dataset with `mlcroissant`.
- Explored its structure and discovered record sets and fields by their `@id`s.
- Loaded a record set into a DataFrame, filtered and normalized a numeric field, and optionally grouped results.
- Visualized critical numeric characteristics.

Further analysis can include statistical testing, correlation analysis, or machine learning, depending on scientific goals and specific data fields available.
